# 🌊 01 — Maritime Vessel Anomaly Detection & Kinematic Outliers
### Models: `vessel_ensemble.joblib`, `isolation_forest.joblib`, `LocalOutlierFactor`

This notebook provides an interactive experimentation environment for detecting anomalous maritime behavior across the Strait of Hormuz.
It explores:
1. **Kinematic Feature Engineering**: Course over ground delta, speed over ground deviation, acceleration variance.
2. **Dual-Path Ensembles**: Spatial Isolation Forest (global density) + Local Outlier Factor (local density).
3. **Probability Calibration**: Isotonic regression mapping raw scores to calibrated anomaly likelihoods.
4. **Slice Analysis**: Assessing performance across Cargo, Oil Tanker, High-Speed Craft, and Fishing vessels.


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT / "service" / "ml-service"))
sys.path.insert(0, str(PROJECT_ROOT / "mlops"))

print(f"Project root resolved: {PROJECT_ROOT}")


In [ ]:
# Load existing production vessel model bundle
model_path = PROJECT_ROOT / "service" / "ml-service" / "models" / "vessel_ensemble.joblib"
if model_path.exists():
    vessel_bundle = joblib.load(model_path)
    print("Successfully loaded production vessel ensemble bundle:")
    for k in vessel_bundle.keys():
        print(f"  - {k}: {type(vessel_bundle[k])}")
else:
    print("Model bundle not found at path; check service/ml-service/models/")


In [ ]:
# Generate Synthetic AIS Trajectory Sample for Testing
from lib.dataset_generator import generate_synthetic_ais_dataset

df_ais = generate_synthetic_ais_dataset(n_samples=500, anomaly_ratio=0.05, random_state=42)
print(f"Generated {len(df_ais)} test records with columns: {list(df_ais.columns)}")
df_ais.head()


In [ ]:
# Run Inference with Ensemble Model
from lib.scoring import score_vessel_features

if "model" in vessel_bundle:
    scores = []
    for idx, row in df_ais.iterrows():
        feat_vector = row[["speed_knots", "course_deg", "course_delta", "speed_delta", "dist_to_tss_km"]].values
        # Predict
        res = vessel_bundle["model"].predict([feat_vector])
        scores.append(res[0])
    df_ais["pred_anomaly"] = scores
    print(f"Anomaly counts: {df_ais['pred_anomaly'].value_counts().to_dict()}")
